# ARC-AGI-3 Socratic Agent

Pipeline completo: **Perceive → Socratic → LLM → Act → Learn**

Sistema autónomo de resolución de juegos ARC-AGI-3 usando:
- **Socratic Engine** como memoria de decisiones
- **Pattern Memory** (59 patrones de VSM-ARC-RSI)
- **Algedonic Feedback** (señales dolor/placer)
- **State Canon** (drift detection)

## 1. Arquitectura del Pipeline

```
┌─────────────────────────────────────────────────────────┐
│                    GAME ENGINE                          │
│  (arc-agi-3: 25 juegos, grid-based)                    │
└─────────────────────────────────────────────────────────┘
                          ↓
┌─────────────────────────────────────────────────────────┐
│  1. PERCEIVE  │ Grid state, actions disponibles        │
├─────────────────────────────────────────────────────────┤
│  2. CANON     │ Predecir vs observar (drift)           │
├─────────────────────────────────────────────────────────┤
│  3. MEMORY    │ Detectar patrones aplicables           │
├─────────────────────────────────────────────────────────┤
│  4. FEEDBACK  │ Señales dolor/placer de historial      │
├─────────────────────────────────────────────────────────┤
│  5. SOCRATIC  │ Sugerencia UCB1 (explotar/explorar)    │
├─────────────────────────────────────────────────────────┤
│  6. LLM       │ Proponer acción con contexto           │
├─────────────────────────────────────────────────────────┤
│  7. ACT       │ Ejecutar en game engine                │
├─────────────────────────────────────────────────────────┤
│  8. LEARN     │ Registrar outcome para próximo paso    │
└─────────────────────────────────────────────────────────┘
```

In [ ]:
# Setup: importar módulos
import sys
sys.path.insert(0, '/home/rmw3/vOSlab-scratch/arc3-socratic/src')

from socratic_agent import SocraticAgent, ActionRecord
from arc3_memory import PatternMemory, Pattern
from arc3_feedback import AlgedonicFeedback, SignalType
from arc3_canon import StateCanon
import json

print('✓ Todos los módulos importados correctamente')

## 2. Demo: Socratic Decision Memory

El agente socrático usa **UCB1** (Upper Confidence Bound) para balancear exploración vs explotación.

In [ ]:
# Demo: Socratic Agent memory
socratic = SocraticAgent('demo-game')
actions = ['ACTION1', 'ACTION2', 'ACTION3', 'ACTION4', 'ACTION5']

# Simulate 10 steps with outcomes
import random
random.seed(42)

for i in range(10):
    state = 'PLAYING'
    hint = socratic.suggest_action(state, actions)
    
    # Simulate outcome (ACTION3 wins 30% of time)
    won = (hint == 'ACTION3' and random.random() < 0.3)
    game_over = (not won and random.random() < 0.1)
    
    record = ActionRecord(
        action=hint or 'ACTION1',
        state_before='PLAYING',
        state_after='WIN' if won else ('GAME_OVER' if game_over else 'PLAYING'),
        levels_before=0,
        levels_after=1 if won else 0,
        game_over=game_over,
        won=won,
        timestamp=0.0,
        pleasurable=won,
        painful=game_over,
    )
    socratic.record_action(record)
    print(f'Step {i+1}: {hint} → {"WIN" if won else ("GAME_OVER" if game_over else "continue")}')

print(f'\n{socratic.get_context_summary()}')

## 3. Demo: Pattern Memory

59 patrones de VSM-ARC-RSI como contexto para el LLM.

In [ ]:
# Demo: Pattern Memory
memory = PatternMemory('demo-game')

# Detect patterns in a sample grid
sample_grid = {
    'grid': [[0, 0, 1, 0], [0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 2]],
    'state': 'PLAYING'
}

detected = memory.detect_patterns(sample_grid)
print(f'Patrones detectados: {detected}')

relevant = memory.get_relevant_patterns(detected, top_k=3)
print(f'\nRelevantes:')
for p in relevant:
    print(f'  {p.to_prompt_fragment()}')

print(f'\n{memory.to_prompt_fragment(detected)}')

## 4. Demo: Algedonic Feedback

Señales de dolor/placer guían la exploración del agente.

In [ ]:
# Demo: Algedonic Feedback
feedback = AlgedonicFeedback('demo-game')

# Simulate game progression
transitions = [
    ('PLAYING', 'PLAYING', 0, 0, 'ACTION1'),
    ('PLAYING', 'PLAYING', 0, 0, 'ACTION2'),
    ('PLAYING', 'PLAYING', 0, 0, 'ACTION3'),
    ('PLAYING', 'GAME_OVER', 0, 0, 'ACTION4'),
    ('PLAYING', 'PLAYING', 0, 1, 'ACTION3'),  # Level progress!
    ('PLAYING', 'WIN', 1, 2, 'ACTION3'),       # WIN!
]

for state_b, state_a, lvl_b, lvl_a, action in transitions:
    signal = feedback.evaluate(state_b, state_a, lvl_b, lvl_a, action)
    emoji = '✓' if signal.signal_type == SignalType.PLEASURE else ('✗' if signal.signal_type == SignalType.PAIN else '→')
    print(f'{emoji} {state_b} → {state_a} | {signal.message}')

print(f'\n{feedback.to_prompt_fragment()}')
print(f'\nRecompensa total: {feedback.get_reward():.2f}')

## 5. Demo: State Canon

Detecta drift entre lo que predice el agente vs lo que realmente ocurre.

In [ ]:
# Demo: State Canon
canon = StateCanon('demo-game')

# Simulate predictions vs observations
scenarios = [
    ({'state': 'PLAYING', 'grid': [[1,2],[3,4]], 'levels_completed': 0},
     {'state': 'PLAYING', 'grid': [[1,2],[3,4]], 'levels_completed': 0}),  # Correct
    ({'state': 'PLAYING', 'grid': [[1,2],[3,4]], 'levels_completed': 0},
     {'state': 'WIN', 'grid': [[1,2],[3,4]], 'levels_completed': 1}),  # Drift!
    ({'state': 'PLAYING', 'grid': [[1,2],[3,4]], 'levels_completed': 1},
     {'state': 'PLAYING', 'grid': [[5,6],[7,8]], 'levels_completed': 1}),  # Drift!
]

for i, (declared, observed) in enumerate(scenarios):
    pred = canon.predict(declared)
    obs = canon.observe(observed)
    snapshot = canon.compare(pred, obs)
    status = '⚠ Drift' if snapshot.drift_detected else '✓ Correct'
    print(f'Scenario {i+1}: {status}')
    if snapshot.drift_detected:
        print(f'  Details: {snapshot.drift_fields}')

print(f'\n{canon.to_prompt_fragment()}')

## 6. Pipeline Integrado: LLM Prompt

Cómo se combina todo en el prompt para el LLM.

In [ ]:
# Demo: Integrated prompt construction
from socratic_agent import SocraticAgent
from arc3_memory import PatternMemory
from arc3_feedback import AlgedonicFeedback
from arc3_canon import StateCanon

# Create modules
socratic = SocraticAgent('ls20')
memory = PatternMemory('ls20')
feedback = AlgedonicFeedback('ls20')
canon = StateCanon('ls20')

# Get contexts
detected = memory.detect_patterns({'grid': [[0,0,1,0],[0,0,0,0]], 'state': 'PLAYING'})
pattern_ctx = memory.to_prompt_fragment(detected)
feedback_ctx = feedback.to_prompt_fragment()
canon_ctx = canon.to_prompt_fragment()
socratic_hint = socratic.suggest_action('PLAYING', ['ACTION1','ACTION2','ACTION3','ACTION4','ACTION5','ACTION6','ACTION7'])

# Build prompt
prompt = f"""Game "ls20" — Step 5
Current: state=PLAYING, grid=10x10, levels=0
Available: ACTION1-ACTION7

Recently tried: {{'ACTION1': 2, 'ACTION3': 1}}
TRY A DIFFERENT ACTION if possible.

Suggestion: {socratic_hint}

{pattern_ctx}

{feedback_ctx}

{canon_ctx}

Choose ONE action (ACTION1-ACTION7). Reply ONLY with action name."""

print('=== LLM PROMPT ===')
print(prompt)

## 7. Resultados Comparativos

| Métrica | Random | Socratic + LLM |
|---------|--------|----------------|
| FPS | 0.10 | 0.55 |
| Acciones únicas (10 steps) | 5/7 | 6/7 |
| Memoria | Ninguna | UCB1 + Patrones |
| Contexto LLM | Ninguno | 59 patrones + feedback |

### Notas
- **CPU-only** (Ollama qwen3:1.7b) — sin GPU
- **Sin entrenamiento** — exploración ciega
- **Próximo paso**: Kaggle GPU (RTX 6000) + modelo grande (Qwen 3.6 27B)

## 8. Estructura del Proyecto

```
arc3-socratic/
├── notebooks/
│   └── demo.ipynb           ← Este notebook
├── src/
│   ├── socratic_agent.py    ← Decision tree + UCB1 memory
│   ├── arc3_memory.py       ← Pattern detection (59 patterns)
│   ├── arc3_feedback.py     ← Algedonic pain/pleasure signals
│   ├── arc3_canon.py        ← State canon drift detection
│   └── my_agent_socratic.py ← Full integrated agent
├── results/
│   └── (evaluation results)
├── docs/
│   └── (architecture docs)
└── README.md
```

## 9. Deployment

### LXC 114 (Desarrollo)
- Python 3.12 (Miniconda)
- Ollama + qwen3:1.7b
- arc-agi 0.9.9
- Starter kit ARC-AGI-3

### Kaggle (Competencia)
- GPU: T4, P100, o RTX 6000 96GB
- Modelos: Qwen 3.6 27B, Gemma 4 31T
- Submission: `make submit` → notebook spliced from my_agent.py